# Phase 1 multi-task transfer learning - Qwen 2.5-0.5B on GSM8K + MATH

Three training runs and one comparison. The headline question:

> **Does combined training beat specialist training, and by how much?**

We train three arms of `Qwen/Qwen2.5-0.5B`, each for the same number of optimiser steps with the same hyperparameters; the ONLY thing that varies is the training-data composition.

| Arm | Training data |
|-----|---------------|
| Combined  | GSM8K + MATH |
| GSM8K-only | GSM8K |
| MATH-only  | MATH  |

After training, each arm is evaluated on **both** test sets. That gives us a 3-arm x 2-test-set grid of mean masked-CE losses; the bar chart at the end is a direct visual answer to the headline question.

**Loss-based eval, not accuracy.** This notebook reports the same metric the Trainer optimises: mean masked cross-entropy on the response tokens of each test example. Generation-based accuracy (parse the model's output, programmatically verify the numerical answer) is a future workstream. Loss is a reasonable proxy for "the model has learned the distribution"; accuracy is what we ultimately care about, but it requires the eval harness from a downstream PRD.

**Approximate runtime.** ~500 steps x ~3 sec/step x 3 arms = ~75 minutes on T4, plus ~10 minutes for evaluation across six (arm, test-set) pairs. Plan for a single ~90-minute Colab session.

**Before running:** set the runtime to T4 GPU. *Runtime -> Change runtime type -> T4 GPU.*

---


## Step 1 - clone the repo and install dependencies

Same setup as the headline notebook. Editable install of the `finpost` package, then a `sys.path` workaround for hatchling's editable-install behaviour on Colab Python 3.12.


In [ ]:
!git clone https://github.com/shannan-liu1/finpost.git
%cd finpost
!pip install -q -e .

In [ ]:
import sys

# /content/finpost is where the clone above lands on Colab; src/ is the
# package root inside the repo. Insert at index 0 so this entry wins over
# any conflicting one Colab might have. Same pattern as the headline
# notebook; needed because of a hatchling editable-install quirk on
# Colab Python 3.12.
sys.path.insert(0, '/content/finpost/src')

import finpost
import finpost.training.trainer  # noqa: F401  - proves submodule resolution works
print('finpost importable from:', finpost.__file__)

## Step 2 - imports and GPU check

`torch.cuda.is_available()` should be True on a T4 runtime. If it's False, switch the runtime and re-run from the top.


In [ ]:
import os
from pathlib import Path

import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:        ', torch.cuda.get_device_name(0))
    print('VRAM (GB):     ', torch.cuda.get_device_properties(0).total_memory / 1e9)

# Optional: load a wandb API key from Colab Secrets so the runs upload
# live. The Trainer auto-detects this; if there's no key, it logs offline
# and you can sync later with `wandb sync wandb/offline-run-*`.
try:
    from google.colab import userdata  # type: ignore
    key = userdata.get('WANDB_API_KEY')
    if key:
        os.environ['WANDB_API_KEY'] = key
        print('WANDB_API_KEY loaded from Colab Secrets.')
except Exception:
    pass

## Step 3 - the three configs

Each arm has its own YAML in `experiments/`:

- `experiments/multitask_combined_qwen.yaml` - `data.sources: [gsm8k, math]`
- `experiments/multitask_gsm8k_only_qwen.yaml` - `data.sources: [gsm8k]`
- `experiments/multitask_math_only_qwen.yaml` - `data.sources: [math]`

All three share the same `lr`, `max_steps`, `warmup_steps`, `per_device_batch_size`, `grad_accum_steps`, `max_seq_len`, `seed`. The whole point of holding everything but the data fixed is that any difference we see in the eval grid can be attributed to data composition, not to a confounder like learning rate.

Each arm writes to its own directory under `results/checkpoints/<run_name>/` so the three runs don't clobber each other. Trainer's retention policy keeps the last 2 checkpoints; for this notebook we only need the final one.


## Step 4 - train Arm 1: COMBINED (GSM8K + MATH)

This is the production-recipe arm. The Trainer logs train/loss, train/lr, train/grad_norm, train/tokens_per_sec, and val/loss to wandb under project `finpost-phase1-multitask`.


In [ ]:
!python -m finpost.training.train --config experiments/multitask_combined_qwen.yaml --device cuda

## Step 5 - train Arm 2: GSM8K-ONLY

Specialist arm for the easier benchmark. Same hyperparameters as Arm 1; only `data.sources` changes.


In [ ]:
!python -m finpost.training.train --config experiments/multitask_gsm8k_only_qwen.yaml --device cuda

## Step 6 - train Arm 3: MATH-ONLY

Specialist arm for the harder benchmark.


In [ ]:
!python -m finpost.training.train --config experiments/multitask_math_only_qwen.yaml --device cuda

## Step 7 - evaluation: 3 arms x 2 test sets

Now the comparison. For each of the three trained arms, we evaluate on **both** the GSM8K test split (~1,319 problems) and the MATH test split (~5,000 problems).

The metric is **mean masked cross-entropy loss** on the response tokens. This is the same loss the Trainer minimises during training, applied to held-out test examples. Lower is better.

**Loss-only by design.** Generation-based accuracy (sample with `model.generate`, parse the answer, verify with the dataset-specific final-answer extractor) is the eval-harness workstream's job. We do not run it here; doing it well requires decoding strategy, batched generation, and per-dataset answer parsers - all of which deserve their own focused implementation.

**Why we build the test loader inline rather than reusing `make_loaders`.** `PhasedSFTDataset` hardcodes `split='train'` and val-splits internally. The test split is a different dataset object entirely. We construct it inline below using the same public primitives (`load_gsm8k`, `load_math`, `serialize_prompt`, `serialize_response`, `PackingCollator`). About 30 lines of code, deliberately visible in the notebook for pedagogical reasons.


In [ ]:
from pathlib import Path
from typing import Any

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from finpost.data.gsm8k import load_gsm8k
from finpost.data.math_dataset import load_math
from finpost.data.schema import Example, Source
from finpost.training.checkpoint import load_checkpoint
from finpost.training.dataset import (
    PackingCollator,
    serialize_prompt,
    serialize_response,
)
from finpost.training.masking import IGNORE_INDEX
from finpost.training.sft import compute_masked_ce_loss


# ---------- inline test-set Dataset ----------------------------------
# Mirrors PhasedSFTDataset.__getitem__'s tuple shape so PackingCollator
# accepts it without modification: (input_ids: tensor, prompt_length: int,
# source: str). The only reason this isn't part of PhasedSFTDataset is
# that PhasedSFTDataset is hardwired to the train split + val carve-out;
# we want the held-out test split, which is loaded separately.
class TestSetDataset(Dataset):
    def __init__(self, examples: list[Example], tokenizer: Any) -> None:
        self.examples = examples
        self.tokenizer = tokenizer

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int, Source]:
        ex = self.examples[idx]
        prompt_text = serialize_prompt(ex.prompt)
        response_text = serialize_response(ex.response)
        prompt_ids = self.tokenizer(prompt_text, add_special_tokens=False)['input_ids']
        response_ids = self.tokenizer(response_text, add_special_tokens=False)['input_ids']
        return (
            torch.tensor(list(prompt_ids) + list(response_ids), dtype=torch.long),
            len(prompt_ids),
            ex.source,
        )


def build_test_loader(
    examples: list[Example],
    tokenizer: Any,
    *,
    max_seq_len: int = 1024,
    batch_size: int = 4,
) -> DataLoader:
    collator = PackingCollator(
        max_seq_len=max_seq_len,
        eos_token_id=tokenizer.eos_token_id,
        isolate_documents=True,
        pad_token_id=tokenizer.pad_token_id or 0,
    )
    return DataLoader(
        TestSetDataset(examples, tokenizer),
        batch_size=batch_size,
        shuffle=False,  # determinism for reproducible eval
        collate_fn=collator,
    )


# ---------- evaluation loop ------------------------------------------
@torch.no_grad()
def eval_loss(model: torch.nn.Module, loader: DataLoader, device: torch.device) -> float:
    '''Mean masked-CE loss over a packed test loader.'''
    model.eval()
    total = 0.0
    n = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        # SDPA needs bool/float, not int64 - same cast as Trainer._forward_loss.
        attention_mask = batch['attention_mask'].to(device).bool()
        position_ids = batch['position_ids'].to(device)
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
        )
        loss = compute_masked_ce_loss(outputs.logits, labels)
        total += float(loss.detach().float().item())
        n += 1
    return total / n if n > 0 else 0.0


# ---------- find each arm's final checkpoint ------------------------
ARMS = {
    'combined':   'results/checkpoints/multitask-combined-qwen-500m',
    'gsm8k-only': 'results/checkpoints/multitask-gsm8k-only-qwen-500m',
    'math-only':  'results/checkpoints/multitask-math-only-qwen-500m',
}


def latest_step_dir(save_dir: str) -> Path:
    root = Path(save_dir)
    candidates = sorted(p for p in root.iterdir() if p.is_dir() and p.name.startswith('step-'))
    if not candidates:
        raise FileNotFoundError(f'No step-*/ checkpoint found under {save_dir}')
    return candidates[-1]  # alphabetical = numerical because zero-padded


# ---------- build the two test loaders once -------------------------
# Tokenizer is shared across arms (same base model). Loaders are
# deterministic and reusable across the three arms.
TOKENIZER_ID = 'Qwen/Qwen2.5-0.5B'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading test sets...')
gsm8k_test = load_gsm8k(split='test')
math_test = load_math(split='test')
print(f'  GSM8K test: {len(gsm8k_test)} examples')
print(f'  MATH test:  {len(math_test)} examples')

gsm8k_loader = build_test_loader(gsm8k_test, tokenizer)
math_loader = build_test_loader(math_test, tokenizer)


# ---------- the 3 x 2 evaluation grid -------------------------------
# results[arm][test_set] = mean masked-CE loss
results: dict[str, dict[str, float]] = {}
for arm_name, save_dir in ARMS.items():
    print(f'\nLoading checkpoint for arm: {arm_name}')
    ckpt_path = latest_step_dir(save_dir)
    state = load_checkpoint(ckpt_path)

    # Fresh model per arm. Loading a state dict on top of the previous
    # arm's parameters would work too, but a fresh from_pretrained is
    # less surprising and the few extra seconds are insignificant
    # against the training time we just spent.
    model = AutoModelForCausalLM.from_pretrained(
        TOKENIZER_ID,
        dtype=torch.bfloat16,
        use_safetensors=True,
    ).to(device)
    # strict=False mirrors Trainer._load_resume - safetensors drops one
    # key of any tied weight pair; the freshly-constructed model has the
    # tying in place so the missing key is harmless.
    model.load_state_dict(state.model_state_dict, strict=False)

    print(f'  evaluating on GSM8K test ({len(gsm8k_test)} examples)...')
    loss_gsm8k = eval_loss(model, gsm8k_loader, device)
    print(f'    -> {loss_gsm8k:.4f}')

    print(f'  evaluating on MATH test  ({len(math_test)} examples)...')
    loss_math = eval_loss(model, math_loader, device)
    print(f'    -> {loss_math:.4f}')

    results[arm_name] = {'gsm8k': loss_gsm8k, 'math': loss_math}

    # Free VRAM before loading the next arm.
    del model
    torch.cuda.empty_cache()


# ---------- print the 3 x 2 table -----------------------------------
print('\n' + '=' * 50)
print('Mean masked-CE loss per (arm, test set):')
print('=' * 50)
print(f'{"arm":<14} | {"GSM8K":<8} | {"MATH":<8}')
print('-' * 36)
for arm_name in ['combined', 'gsm8k-only', 'math-only']:
    g = results[arm_name]['gsm8k']
    m = results[arm_name]['math']
    print(f'{arm_name:<14} | {g:<8.4f} | {m:<8.4f}')


## Step 8 - bar chart of the 3 x 2 grid

Two bars per arm. Lower is better. The visual story we expect:
- The combined arm should be reasonable on both test sets.
- Each specialist should be better on its own test set than the combined arm.
- The harder question: is each specialist worse on the *other* test set than the combined arm? If yes, that's the transfer-learning story we set out to measure.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

arms = ['combined', 'gsm8k-only', 'math-only']
gsm8k_vals = [results[a]['gsm8k'] for a in arms]
math_vals = [results[a]['math'] for a in arms]

x = np.arange(len(arms))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width / 2, gsm8k_vals, width, label='GSM8K test')
b2 = ax.bar(x + width / 2, math_vals, width, label='MATH test')

ax.set_xticks(x)
ax.set_xticklabels(arms)
ax.set_ylabel('Mean masked-CE loss (lower is better)')
ax.set_title('Phase 1 multi-task transfer: 3 arms x 2 test sets')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Numerical labels above each bar.
for bars in (b1, b2):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f'{height:.3f}',
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=9,
        )

plt.tight_layout()
plt.show()

## Step 9 - interpretation prompts

Look at the chart above and answer for yourself before reading further:

1. **Which arm wins on each test set?** Specialist on its own domain is the expected best; the size of the win matters more than the direction.
2. **Specialist gap on the *other* test set.** GSM8K-only on MATH, and MATH-only on GSM8K - how much worse than the combined arm? A large gap means the specialist forgot transferable skills; a small gap means GSM8K and MATH are similar enough that one buys you most of the other.
3. **Is combined ever the *winner* on a single test set?** Unlikely (specialists usually win their own domain), but a small gap means combined-training is essentially free transfer learning.

**Caveats before drawing strong conclusions:**

- 500 optimiser steps is a smoke run. Train differences on this scale are noisy. The headline conclusion should come from a 3000-step replication using `experiments/baseline.yaml` as the template (see "Next steps" below).
- Loss != accuracy. A small loss gap can correspond to a much larger accuracy gap once we evaluate by generating answers and parsing them. Treat the loss numbers above as "roughly proportional to accuracy" but not as the final answer.
- Token-share imbalance. MATH problems are longer than GSM8K problems on average, so the combined arm sees a slight bias toward MATH (~40/60 GSM8K/MATH split by tokens). This is documented in `phase1-sft-trainer/PRD.md` decision Q-D - per-source weighting was deliberately rejected in favour of natural mixing.

## Next steps

If the smoke run looks reasonable:

1. **Longer runs** - copy each multitask config, raise `max_steps` to 3000 (the published baseline), and re-run all three. ~6 hours on T4 total.
2. **Generation accuracy eval** - blocked on the eval-harness workstream. Once that lands, run it on each of these three checkpoints.
3. **Per-source val curves** - the Trainer logs one combined `val/loss`. Splitting it into per-source val losses would let us see *during* training whether each arm is improving on the test sets it should care about. Filed as a follow-up issue, not in this notebook.
